# Глава 4. Модель трансформера
В прошлой главе мы рассмотрели архитектуру рекуррентных сетей, которые на протяжении почти 20 лет были основным инструментом для кодирования текстовых данных. Обсудили их преимущества и недостатки. 

В этой главе поговорим о Трансформере - модели, описанной в 2017 году и совершившей революцию в языковых моделях. Благодаря прорыву на бенчмарках, архитекутра быстро стала стандартом для всех языковых моделей и остается им по сей день

## Мотивация
До 2017 года доминирующим подходом к обработке последовательностей были рекуррентные сети (RNN) и конкретно два их варианта — **LSTM** сеть и **GRU** сеть, которые читают последовательность шаг за шагом, поддерживая скрытое состояние: $h_t = f(h_{t-1}, x_t)$. У этой схемы три фундаментальные недостатка.

1) Первая проблема — вычисления в такой модели строго последовательны, состояние $h_t$ нельзя посчитать, пока не рассчитано $h_{t-1}$, поэтому обучение не распараллеливается по длине последовательности: текст из $n$ токенов требует $n$ последовательных шагов. возможности GPU не задействованы

2) Вторая проблема - неустойчивость обучения. Градиент, который на обучении проходит назад через десятки шагов, либо затухает (vanishing gradient), либо взрывается (exploding gradient), сложно поддерживать балансировку. Гейты, появившиеся в LSTM и GRU, смягчают проблему, но не убирают её: путь сигнала между токенами $i$ и $j$ по-прежнему имеет длину $|i-j|$

3) Третья проблема — бутылочное горлышко ограничивает возможности генеративные моделей. В классической схеме **seq2seq** [(Sutskever et al., 2014)](https://arxiv.org/pdf/1409.3215) энкодер сжимает всё входное предложение в один вектор, из которого декодер порождает выход. Для длинных предложений одно фиксированное представление оказывается слишком тесным<br><br><img src="img/seq2seq1.png" width=500><br><br>Эту проблему решил механизм внимания [(Bahdanau et al., 2014)](https://arxiv.org/abs/1409.0473): декодер на каждом шаге строит взвешенную комбинацию всех скрытых состояний энкодера, то есть напрямую «смотрит» на нужные части входа. Внимание резко улучшило перевод, но оставалось надстройкой над рекуррентной сетью — первые две проблемы никуда не делись

<img src="img/attention1.png" width=200>

В 2017 году группа исследователей из Google выпустила статью "Attention Is All You Need", в которой они описали архитектуру **Transformer** [(Vaswani et al., 2017)](https://arxiv.org/abs/1706.03762). Авторы предложили радикально пересмотреть подход к моделированию текста и полностью убрать рекуррентность, оставив только механизм внимания. Тогда посчитать связь между любыми двумя токенами теперь можно за константное время $O(1)$, так что дальние зависимости моделируются напрямую и значит все позиции последовательности можно обработать за один подход, что идеально ложится на возможности GPU/TPU и открывает дорогу масштабированию — главному секрету успеха моделей данного класса

## Самовнимание
Центральный компонент архитектуры трансофрмера - это механизм самовнимания. Напомним, что внимание, появившееся несколькими годами ранее в контексте seq2seq моделей перевода - это способ учесть сигнал сразу от всех токенов.

Но если в рекуррентных моделях внимание связывало декодер с энкодером, то самовнимание не делает различий. **Self-attention** — операция, в которой каждый токен создает своё представление, собирая информацию со всех токенов последовательности. 

<img src="img/self_attention3.png" width=300>

Вычисление самовнимания можно описать тремя шагами

Сначала каждый токен $x$ переводится в три представления с помощью обучаемых проекций. Эти представления называют: 
- query $q = W_Q x$ («что я ищу»),
- key $k = W_K x$ («по какому признаку меня можно найти»)
- value $v = W_V x$ («какую информацию я отдаю»).

Важно понимать, что эти роли (key, query, value) условные, модель сама определяет, что и как именно кодировать в этих векторах.

Далее каждый вектор q сопоставляется со всеми векторами k путем вычисления скалярного произведения между ними, и получаемый вектор нормируется через функцию softmax.

В матричной форме выглядит так:
$$
\mathrm{Attention}(Q, K, V) = \mathrm{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right)V
$$



Нормировка величиной $\sqrt{d_k}$ важна: дисперсия скалярного произведения растёт линейно с увеличением размерности $d_k$, поэтому без масштабирования softmax уходит в насыщение, а градиенты почти обнуляются.

На практике вместо одной операции внимания используется несколько инстансов. Такой механизм называют **multi-head attention**: берем $h$ эксземпляров, каждый со своими проекциями меньшей размерности $d_k = d_{model}/h$, и считаются параллельно. Затем их выходы конкатенируются и смешиваются выходной матрицей $W_O$. Идея в том, что разные головы обучаются отслеживать разные типы отношений — это могут быть, например, синтаксические связи, соседние позиции, кореференцию. По суммарному числу параметров $h$ голов эквивалентны одной «широкой» голове, так что многоголовость почти бесплатна.

<img src="img/self_attention2.png" width=200>

## Общая архитектура
Оригинальный трансформер — это encoder-decoder для машинного перевода: энкодер строит представление входного предложения, декодер порождает выходное, подглядывая в энкодер.

Энкодер — стопка из $N$ одинаковых блоков. Блок состоит из двух подслоёв: self-attention и позиционно-независимой полносвязной сети **FFN** (feed-forward network). FFN применяется к каждому токену отдельно и одинаково:

$$
\mathrm{FFN}(x) = W_2\, f(W_1 x + b_1) + b_2, \qquad d_{ff} = 4\,d_{model}
$$

В оригинале $f$ — ReLU; в современных моделях чаще **GELU** (Hendrycks & Gimpel, 2016) или гейтированный вариант **SwiGLU** (Shazeer, 2020). Если attention — это обмен информацией между позициями, то FFN — её обработка «на месте»; на FFN приходится примерно две трети параметров блока. Каждый подслой обёрнут в residual-связь и нормализацию (о них ниже).

<img src="img/transformer_ffn.png" width=300>

Декодер устроен похоже, но в блоке три подслоя: self-attention с каузальной маской (декодер не должен видеть будущее — см. раздел про обучение), затем **cross-attention**, в котором query берутся из декодера, а key и value — из выхода энкодера (прямой наследник внимания Bahdanau), затем FFN.

На входе токены превращаются в векторы таблицей эмбеддингов и складываются с позиционными кодированиями. На выходе декодера стоит линейный слой в размер словаря и softmax, дающий распределение следующего токена. Часто применяется **weight tying** [(Press & Wolf, 2016)](https://arxiv.org/abs/1608.05859): выходная матрица совпадает с транспонированной матрицей эмбеддингов, что экономит параметры и слегка улучшает качество.

Конфигурация оригинальной base-модели: 6 блоков энкодера и 6 декодера, $d_{model}=512$, $h=8$, $d_{ff}=2048$, около 65M параметров; big-версия — $d_{model}=1024$, $h=16$, около 213M.

## Позиционное кодирование
Напомним, что в центре самовнимания находится расчет матрицы $QK^T$. Легко заметить, что порядок токенов в такой механике никак не влияет на выходное значение: если перемешать токены на входе, выходы перемешаются точно так же, и с точки зрения модели текст «cat ate mouse» станет неотличим от «mouse ate cat»

Нам же хочется учитывать порядок токенов в тексте, ведь порядок в значительной степени определяет семантику => надо научиться как-то добавлять в сигнал от токена информацию о его позиции. Иначе мы откатываемся к старым bag-of-words моделям

Позиционные фичи можно добавлять в разные места, кодироваться детерминированно или обучаемо

В оригинальной работе "Attenion is all you need" информация о позиции просто прибавляется перед первым слоем Трансформера к оригинальному эмбедингу. Прибавка имеет ту же размерность что и сам эмбединг

$$x' = x + PE$$

Выпишем, как будет вычисляться Attention между двумя токенами $x_m$ и $x_n$ с учетом информации об их позиициях:
$$q_m^\top k_n = (W_Q x_m)^T (W_K x_n) = (W_Q (x_m+PE_m))^T (W_K (x_m+PE_n))$$

Если раскроем скобки, увидим, что величина скалярного произведения складывается из 4 составляющих:
$$q_m^\top k_n = \underbrace{x_m^\top W_Q^\top W_K x_n}_{\text{чисто токенный сигнал}} + \underbrace{x_m^\top W_Q^\top W_K PE_n}_{\text{позиционная прибавка}} + \underbrace{PE_m^\top W_Q^\top W_K x_n}_{\text{позиционная прибавка}} + \underbrace{PE_m^\top W_Q^\top W_K PE_n}_{\text{чисто позиционный сигнал}}$$

Хочется, чтобы позиционная прибавка $PE_m^\top W_Q^\top W_K PE_n$ не зависела от абсолютной позиции токена, а зависела только от расстояния между токенами, поскольку это вполне в логике языка - какая разница, находятся ли сравниваемые токены в начале  или в конце промпта, важно, на каком они расстоянии друг от друга. Большинство способов кодирования из списка ниже реализованы так, чтобы это свойство выполнялось (строго или приближенно)

**Синусоидальные** [(Vaswani et al., 2017)](https://arxiv.org/abs/1706.03762) <br>Позиционный вектор детерминирован и составлен из синусов и косинусов разных частот. Он прибавляется к эмбеддингу токена $$PE(pos) = \bigg[\sin(pos \cdot \omega^1), \cos(pos \cdot \omega^1), \,\, \sin(pos \cdot \omega^2),  \cos(pos \cdot \omega^2) \,\, ... \,\, \sin(pos \cdot \omega^{d/2}), \cos(pos \cdot \omega^{d/2}) \bigg] \text{\,, где}$$ 
$$ \text{где \,} \omega = \frac{1}{10000^{2/d}} \text{ - частота}$$  

Почему выбрана именно такая механика:
- эмбединги не повторяются, вектор каждой позиции 1..N уникален
- скалряное произведение эмбедингов зависит от расстояния, но не зависит от абсолюта

**Обучаемые**<Br> каждая позиция $1..N$ кодируется своим эмбедингом (той же размерности что эмбединги токенов), затем просто складывается с эмбедингом токена. Так было сделано, например, в моделях BERT и GPT-2. Работает хорошо, но главный минус - жёстко ограничивает максимальную длину контекста числом строк таблицы

**Относительные** [(Shaw et al., 2018)](https://arxiv.org/abs/1803.02155) <br>информация о позициях добавляется сразу в произведение $QK^T$ аддитивной прибавкой: $Q(K+PE)^T$. Кодируется прибавка как эмбединг разницы позиций $i - j$. Само кодирование обучаемое $w_1 = [], w_2 = [] ... w_k=[]$ и ограничено: если |i - j| > k, то разницы нет - большие расстояния схлопываются в одно значение. Два варианта: добавлять только к матрице K, добавлять к матрице V тоже

**ALiBi** [(Press et al., 2021)](https://arxiv.org/abs/2108.12409) <br>информация о позициях добавляется сразу к произведению $QK^T$ в виде скалярной прибавки $QK^T + PE$. Кодируется эта прибавка детерминированно как взвешенная разницей позиций $-m(i-j)$, где вес $m = (1/2)^{h}$ свой для каждой из 8 Attention Heads (h=1..8)

$$PE_1 = \left[\frac{1}{2}\right]^1, \quad PE_2 = \left[\frac{1}{2}\right]^2 \quad ... \quad PE^8 = \left[\frac{1}{2}\right]^8$$

Обратите внимание, что прибавка отрицательная - мы штрафуем связку за слишком большое расстояние между токенами

**RoPE** [(Su et al., 2021)](https://arxiv.org/abs/2104.09864) <br>Метод ROPE = Rotary POsition Embedding. На момент 2026 года это стандарт, используется для кодирования позиции токенов в LLaMA, Qwen, DeepSeek и большинство открытых моделей<Br><br>Идея: пары компонент векторов $q$ и $k$ поворачиваются на угол, пропорциональный позиции токена; тогда скалярное произведение $q_i^\top k_j$ автоматически зависит только от разности $i-j$. Абсолютная операция даёт относительный эффект. Дополнительное приятное свойство - контекст обученной модели можно растягивать интерполяцией позиций.



__NoPE__<Br>
польза добавления в модель позиционной информации до сих пор вызывает дискуссии. Например, в исследовании [(Haviv, 2022)](https://arxiv.org/abs/2203.16634) её откровенно челленджат. Авторы назвали свой подход __NoPE__ и показали, что без всяких позиционных эмбедингов модель способна обучаться. Показывается, что она выучивает абсолютные позиции токена по косвенным признакам. А в более позднем исследовании [(Wang et al, 2024)](https://arxiv.org/abs/2404.12224) показали, что модель способна выучивать не только абсолютные, но и относительные позиции

## Residual connections и нормализация

Два важных компонента

Первая - это остаточные связи между слоями (**Residual connections**) [(He et al., 2015)](https://arxiv.org/abs/1512.03385), пришедшая из компьютерного зрения: выход подслоя не заменяет вход, а прибавляется к нему, $y = x + \mathrm{Sublayer}(x)$. 

Во-первых, это короткий путь для градиента: через тождественную ветку он течёт к нижним слоям, не затухая, что и позволяет строить сети из десятков и сотен блоков. 

Во-вторых, это удобная точка зрения на всю архитектуру: сквозь модель идёт **residual stream** — общая «шина» размерности $d_{model}$, из которой каждый подслой читает и в которую дописывает свою поправку (Elhage et al., 2021). Слои не переписывают представление, а инкрементально его уточняют; мы вернёмся к этому в разделе про интерпретацию.

Второй компонент - это обязательная нормализация. Используется послойная нормализация **LayerNorm** [(Ba et al., 2016)](https://arxiv.org/abs/1607.06450) нормализует вектор каждого токена по его признакам: $y = \gamma \odot (x - \mu)/\sigma + \beta$, где $\mu$ и $\sigma$ считаются по компонентам этого конкретного вектора. В отличие от BatchNorm статистики не зависят от батча и длины последовательности, что критично для текста. Нормализация удерживает масштаб активаций и стабилизирует обучение.

Вопрос куда добавлять. В оригинальной арзхитектуре использовался **Post-LN** — нормализация после сложения с residual-веткой. Такая схема даёт хорошее качество, но нестабильна для глубоких моделей и требует аккуратного прогрева learning rate. Начиная с GPT-2 стандартом стал **Pre-LN** [(Xiong et al., 2020)](https://arxiv.org/abs/2002.04745) — нормализация на входе подслоя, $y = x + \mathrm{Sublayer}(\mathrm{LN}(x))$: residual-путь остаётся чистым тождественным, и глубокие модели обучаются заметно стабильнее.

Популярное упрощение — **RMSNorm** [(Zhang & Sennrich, 2019)](https://arxiv.org/abs/1910.07467): вектор лишь делится на среднеквадратичную норму и умножается на $\gamma$, без вычитания среднего и без сдвига $\beta$. Считается быстрее и не ухудшает качество модели на бенчмарках. Использовался например, в модели T5, а также семействе моделей LLaMA.

## Три семейства моделей
Оригинальная архитектура — encoder-decoder, но практика быстро показала, что для многих задач достаточно одной из половин. По тому, какая часть используется и на чём модель предобучается, выделяют три семейства; их сравнение — удобная рамка для всей истории 2018–2020 годов.

### BERT модели
**BERT** (Devlin et al., 2018) используют только энкодер: каждый токен видит весь текст целиком налево и направо. Задача **masked language modeling (MLM)**: 15% токенов случайно выбираются, из них 80% заменяются на [MASK], 10% на случайный токен, 10% остаются как есть, и модель восстанавливает оригинал по контексту с обеих сторон. 

В оригинальном BERT 2018 года дополнительно обучали под задача **next sentence prediction (NSP)** — определить, следуют ли два фрагмента друг за другом. 

BERT закрепил парадигму pretrain → fine-tune: одна дорогая предобученная модель, поверх которой под каждую задачу дообучается лёгкая голова. Стихия encoder-моделей — понимание текста: классификация, NER, extractive QA, ранжирование и эмбеддинги для поиска. Генерировать текст они не умеют — у них нет авторегрессионного разложения.

Модель поставлялась в двух модификациях Base и Large

(Liu et al., 2019) из Facebook решили проверить, исчерпан ли потенциал самой архитектуры BERT, или дело в неоптимальном обучении. Оказалось, что оригинальный BERT был существенно недообучен, и качество можно поднять, просто сконфигурировав модель. Так появилась модель __RoBERTa__. Самое главное авторы отказались от обучения под задачу предсказания следующего предложения (NSP), так как бесполезной и даже вредной, заменили статическое маскирование динамическим (маска генерируется заново при каждой подаче примера, а не один раз при подготовке данных), в 10 раз увеличили объём корпуса с 16 до 160 ГБ, размер батча — до 8000 последовательностей, и заменили токенизатор на байтовый BPE-токенизатор со словарём в 50 тысяч токенов. В результате — превосходство над BERT на GLUE, SQuAD и RACE при идентичной архитектуре, что закрепило RoBERTa в роли стандартного энкодера-

(Google, 2019) обратили внимание, что дальнейшее наращивание BERT упирается в объём памяти GPU, и для развития желательно сократить число параметров. Предложили модель __ALBERT__, несколько оптимизаций. Во-первых, сделали факторизацию матрицы эмбеддингов (токен сначала проецируется в низкоразмерное пространство E, а затем в скрытую размерность H). Во-вторых все параметры сделали разделяемыми (shared) между слоями трансформера, тем самым радикально сократив общее кол-во параметров. И в третьих задачу NSP заменили на задачу SOP (sentence order prediction) — предсказание порядка двух соседних сегментов, которое, в отличие от NSP, нельзя решить простым определением темы.

В итоге размер ALBERT-base в 9 раз меньше BERT-base, однако экономится только память: объём вычислений на прямой проход не сокращается, поэтому крупные конфигурации ALBERT работают медленнее BERT.

[(Sanh et al)](https://arxiv.org/abs/1910.01108) из Hugginface  (Hugging Face, 2019). Хотели получить универсальную минималистичную версию BERT, пригодную для любых downstream-задач, поэтому применили дистилляцию к предобученой BERT модели.  Модель получила название __DistilBERT__. Число слоёв сократили вдвое (посольку глубина влияет на скорость сильнее ширины), студент инициализируется весами каждого второго слоя учителя, были удалены эмбеддинги сегментов и слой пулинга.

В ходе дистилляции оптимизировалась комбинированная функции потерь, состоящая из трёх компонент: KL-дивергенции между распределениями учителя и студента, функция потерь стандартной MLM задачи, а также косинусное расхождение, выравнивающе направления скрытых векторов. В итоге модель на 40% меньше и на 60% быстрее BERT-base при сохранении около 97 % его качества на GLUE.

[(Sun et al)](https://arxiv.org/pdf/2004.02984) разрабатывали компактную BERT подобную модель для выполнения на мобильном устройстве. Решено сохранить глубину BERT модели в 24 слоя, но сделать каждый блок очень узким за счёт замены на двухслойную проекцию бутылочного горлышка, благодаря чему межслойные представления остаются «широкими», а кол-вг вычислений внутри блока резко сокращаются. Модель назвали __MobileBERT__. Напрямую обучить такую узкую глубокую сеть не удаётся, поэтому знания послойно переносятся из специально обученного учителя IB-BERT-large — сопоставляются карты внимания и выходы скрытых слоёв. Нарушенный bottleneck'ом баланс между вниманием и FFN восстанавливается стеком из четырёх полносвязных подслоёв, а ради латентности нормализация заменена на упрощённый NoNorm, а GELU — на ReLU

Также [(Landola et al, 2020)](https://arxiv.org/abs/2006.11316) создавали версию BERT для выполнения на смартфонах и решили исользовать оптимизацию, которая хорошо зарекомендовала себя в компьютерном зрении (а именно в моделях MobileNet и ShuffleNet): позиционно-независимые полносвязные слои трансформера математически эквивалентны одномерным свёрткам с ядром 1. Модель назвали __SqueezeBERT__, чтобы подчеркнуть меньшие размеры сети. Главное, что сделали авторы - заменили слои внимания в BERT групповыми свёртками (convolution). Каналы разбиваются на G групп, каждая обрабатывается независимо, что сокращает вычисления примерно в G раз. Приём применяется для матриц Q, K, V и в полносвязных подслоях. В выходной проекции $W_O$ сохраняется обычная свёртка, так как там важно смешивание всех каналов. Обучение дополняется дистилляцией. Результат — ускорение в 4,3 раза относительно BERT-base на мобильном CPU при качестве, сопоставимом с MobileBERT.

Microsoft не осталась в стороне и в 2020 году предложили свой вариант модели: который назвали __DeBERTa__. При разработке акцент был сделан на способ кодирования позиций: каждый токен описывается двумя векторами, а вес внимания раскладывается на три составляющие — «содержание — содержание», «содержание — позиция» и «позиция — содержание», что позволяет явно моделировать зависимость связи между словами от расстояния между ними. Абсолютные позиции при этом не выбрасываются, а вводятся непосредственно перед выходным softmax-слоем, поскольку без них неразличимы, например, подлежащее и дополнение. DeBERTa первой превзошла человеческий уровень на задачах бенчмарка SuperGLUE (тест на глубокое понимание смысла текста).

В 2024 году [(Warner et al)](https://arxiv.org/abs/2412.13663) решили пересобрать современный энкодер «с нуля», перенеся в него все инженерные наработки, накопленные за шесть лет. Модель назвали __ModernBERT__. что было сделано: абсолютные позиционные эмбеддинги заменили на RoPE, полное внимание применяется лишь в каждом третьем слое, а остальные используют скользящее окно в 128 токенов, что позволило довести длину контекста до 8192 токенов. Добавлены GeGLU-активации и пре-нормализация, убраны почти все смещения (bias), padding-токены не маскируются, а физически удаляются из батча (unpadding + Flash Attention), а глубина и размерности подобраны под эффективную загрузку распространённых GPU. Модель показывает лучшие среди энкодеров результаты на GLUE и в задачах поиска, работая в 2–4 раза быстрее сопоставимых моделей с длинным контекстом.

(Clark et al., 2020) решили моделировать текст в виде корректирующего автоенкодера. Генератор решает MLM . Дискриминатор учится определять, оригинальное слово или заменено генератором. Модель получила название **ELECTRA**. 

### Декодерные модели
**GPT** (Radford et al., 2018) только декодер (без cross-attention): каузальное внимание и классическая задача языкового моделирования — предсказание следующего токена. **GPT-2** (Radford et al., 2019) с 1.5B параметров продемонстрировала, что при достаточном масштабе задачи можно решать zero-shot, просто формулируя их текстом. **GPT-3** (Brown et al., 2020) со 175B параметров открыла **in-context learning**: несколько примеров прямо в промпте настраивают модель на задачу без единого обновления весов. Декодерная ветвь дала универсальный интерфейс — любая задача сводится к продолжению текста — и стала магистральной для современных LLM (**LLaMA** (Touvron et al., 2023), Mistral, Qwen, DeepSeek и другие). Причины доминирования прозаичны: простейшая целевая функция, обучающий сигнал с каждого токена корпуса, генерация «из коробки» и хорошее масштабирование.

### Энкодер-декодерные модели

Сейчас редкая конфигурация, но в период раннего развития Трансформеров встречалась. Идея такая: сначала входной промпт обрабатывается энкодером, а затем декодер начинает авторегрессионную генерацию, при этом имея доступ к получнным энкодреом представлениям. Доступ реализуется через добавление дополнительного Cross-Attention слоя после обычного Attention. Если обычный Attention видит только сгенерированные токены, то Cross-Attention видит только выходные эмбединги промпта

**T5** (Raffel et al., 2019) провозгласила принцип text-to-text: любая задача — перевод строки в строку с текстовым префиксом-инструкцией («translate English to German: ...», «summarize: ...»); предобучение — span corruption, восстановление вырезанных отрезков. 

**BART** (Lewis et al., 2019) — денойзинг-автоэнкодер: текст портится маскированием, перестановкой и удалением фрагментов, модель восстанавливает оригинал; сильный вариант для суммаризации. 

Сегодня ниша encoder-decoder моделей сузилась — декодеры с длинным контекстом покрывают те же задачи, — но архитектура жива, например в распознавании речи (Whisper (Radford et al., 2022)) и переводе

## Обучение

Разберём обучение на главном сегодня случае — decoder-only модели.

### Целевая функция

Языковая модель раскладывает вероятность текста по цепному правилу: $p(x_1, \dots, x_n) = \prod_t p(x_t \mid x_{<t})$. Обучение — максимизация правдоподобия корпуса, то есть минимизация кросс-энтропии предсказания следующего токена:

$$
\mathcal{L} = -\sum_t \log p_\theta(x_t \mid x_{<t})
$$

Экспонента среднего лосса — perplexity, стандартная метрика качества языковых моделей. При обучении используется **teacher forcing**: условием всегда служит настоящий префикс из данных, а не собственные (возможно ошибочные) предсказания модели; это делает обучение стабильным и, как мы сейчас увидим, параллельным.

### Каузальная маска

Ключевая деталь декодера — **каузальная маска**. К матрице логитов внимания перед softmax прибавляется треугольная маска $M$: $M_{ij} = 0$ при $j \le i$ и $M_{ij} = -\infty$ при $j > i$:

$$
\mathrm{Attention}(Q, K, V) = \mathrm{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}} + M\right)V
$$

После softmax веса на будущих позициях становятся нулями: токен видит только себя и то, что левее. Маска решает сразу три задачи. Во-первых, корректность: без неё предсказание «следующего» токена было бы подглядыванием в ответ. Во-вторых, параллелизм обучения: благодаря маске один forward-проход по последовательности из $n$ токенов честно даёт $n$ задач предсказания одновременно — позиция $t$ предсказывает токен $t+1$, не видя его. Модель получает $n$ обучающих сигналов за один проход; это и есть главное вычислительное преимущество перед RNN, где те же $n$ предсказаний требуют $n$ последовательных шагов. В-третьих, согласованность обучения и инференса: представление токена не зависит от того, что стоит правее, поэтому при генерации его не нужно пересчитывать — на этом факте построен KV-cache.

Для сравнения: в энкодере (BERT) маски нет — внимание двунаправленное; в encoder-decoder маска стоит только в self-attention декодера.

### Алгоритм обучения

Один шаг обучения выглядит так:

1. Корпус токенизируется и нарезается на последовательности фиксированной длины, из которых собирается батч $B \times n$.
2. Forward-проход даёт тензор логитов $B \times n \times V$ — распределение следующего токена для каждой позиции.
3. Таргеты — тот же батч, сдвинутый на один токен влево; по логитам и таргетам считается средняя кросс-энтропия.
4. Backward-проход вычисляет градиенты; шаг оптимизатора обновляет веса.

Стандартный оптимизатор — **AdamW** (Loshchilov & Hutter, 2017) с прогревом learning rate и последующим косинусным затуханием, gradient clipping и обучением в смешанной точности. В оригинальной статье также применялись dropout и label smoothing. Тонкости масштабирования такого обучения на тысячи GPU — тема отдельной главы.

## Инференс

### Авторегрессионная генерация

Обученная модель порождает текст токен за токеном:

1. Промпт токенизируется и прогоняется через модель.
2. Логиты последней позиции дают распределение следующего токена.
3. Из распределения выбирается токен (стратегии выбора — в следующем разделе) и дописывается к последовательности.
4. Шаги 2–3 повторяются, пока не выпадет специальный токен конца текста или не исчерпан лимит длины.

Наивная реализация на каждом шаге прогоняет через модель всю удлинившуюся последовательность заново. Это расточительно: из-за каузальной маски представления старых токенов не зависят от новых и при повторных проходах вычисляются в точности такими же.

### KV кэш

**KV-cache** устраняет повторные вычисления. Для каждого слоя сохраняются векторы $K$ и $V$ всех уже обработанных токенов. На очередном шаге модель считает $q$, $k$, $v$ только для одного нового токена, дописывает его $k$ и $v$ в кеш, а внимание вычисляет между query нового токена и всем накопленным кешем. Query старых токенов хранить не нужно — их выходы уже посчитаны и больше не потребуются.

Цена — память. Кеш хранит $2 \cdot L \cdot n \cdot d$ чисел (K и V, на каждый слой, на каждый токен). Для GPT-3 ($L = 96$, $d = 12288$) в fp16 это $2 \cdot 96 \cdot 12288 \cdot 2$ байта $\approx 4{,}5$ МБ на один токен, то есть около 9 ГБ на одну последовательность длины 2048 — сопоставимо с самими весами при большом батче. Отсюда популярные модификации внимания: **multi-query attention (MQA)** (Shazeer, 2019), где все query-головы делят одну пару K/V-голов, и компромиссный **grouped-query attention (GQA)** (Ainslie et al., 2023), где K/V-головы делятся на группы; кеш сокращается на порядок почти без потери качества (LLaMA-2 70B, Mistral и большинство современных моделей).

### Префилл и декодирование

Инференс распадается на две фазы с разным характером. **Prefill** — обработка промпта: все его токены известны заранее и проходят через модель одним параллельным проходом, заполняя KV-cache; фаза упирается в вычисления (compute-bound), а её длительность — это задержка до первого токена. **Decode** — порождение по одному токену: на каждый токен нужно прочитать из памяти все веса модели и весь кеш, так что фаза упирается в пропускную способность памяти (memory-bound), а измеряется в токенах в секунду. Эта асимметрия определяет инженерию инференса: батчирование множества запросов на фазе decode, а также приёмы вроде **speculative decoding** (Leviathan et al., 2022), где маленькая черновая модель предлагает несколько токенов вперёд, а большая проверяет их одним параллельным проходом, похожим на prefill.

## Стратегии выбора следующего токена

Модель выдаёт распределение $p(x_t \mid x_{<t})$ над словарём; отдельный вопрос — как превратить его в конкретный токен. Стратегия выбора не связана с обучением: это интерфейс к готовой модели, и одна и та же модель ведёт себя очень по-разному при разных настройках.

**Greedy decoding** — на каждом шаге берётся argmax. Детерминированно и дёшево, но локально лучший токен не гарантирует глобально хорошего текста, а в открытой генерации жадный выбор вырождается в повторы и зацикливания.

**Beam search** поддерживает $k$ лучших префиксов по суммарной логвероятности, расширяя каждый на каждом шаге. Стандарт для задач с «правильным ответом» — перевода и суммаризации (обычно с поправкой на длину). Для открытой генерации работает плохо: максимально вероятный текст оказывается тусклым и повторяющимся, тогда как человеческий текст регулярно содержит токены-«сюрпризы» (Holtzman et al., 2019).

**Temperature** масштабирует логиты перед softmax: $p_i \propto \exp(z_i / T)$. При $T \to 0$ получаем greedy, $T = 1$ — исходное распределение, $T > 1$ — более равномерное. Это ручка «детерминизм ↔ разнообразие».

**Top-k sampling** (Fan et al., 2018) оставляет $k$ самых вероятных токенов, перенормирует и сэмплирует из них. Недостаток — фиксированное $k$ при изменчивой форме распределения: иногда разумных продолжений два, иногда сотня.

**Nucleus sampling (top-p)** (Holtzman et al., 2019) решает это адаптивно: берётся минимальное множество токенов, чья суммарная вероятность достигает $p$ (например 0.9). Острое распределение сводится к паре кандидатов, плоское — к десяткам. Главный эффект — отсечение длинного хвоста малонадёжных токенов, из-за которого при сэмплировании накапливаются ошибки.

**Min-p sampling** (Nguyen et al., 2024) — более свежий вариант: порог задаётся относительно лидера, остаются токены с $p_i \ge \alpha \cdot p_{max}$; хорошо сохраняет связность при высоких температурах.

**Repetition penalty** (Keskar et al., 2019) дополнительно штрафует логиты уже встречавшихся токенов, грубо подавляя зацикливание.

На практике методы комбинируют: температура плюс top-p — типичный режим диалоговых моделей, а greedy или низкая температура — режим задач с проверяемым ответом (код, извлечение фактов).

## Интерпретация

Что происходит с сигналом внутри стопки блоков? Удобная система координат — уже упоминавшийся residual stream (Elhage et al., 2021): вектор каждой позиции — это «рабочая память», в которую все подслои дописывают свои поправки. Разделение труда такое: attention переносит информацию между позициями («какой токен на что смотрит»), а FFN обрабатывает её на месте; есть данные, что FFN работает как ассоциативная память вида ключ → значение, где нейроны срабатывают на паттерны и дописывают в поток связанные с ними факты (Geva et al., 2020).

По глубине выстраивается иерархия признаков — та же картина, что в свёрточных сетях зрения, где слои идут от краёв и текстур к частям объектов и целым объектам. Probing-исследования показали, что BERT «заново открывает классический NLP-пайплайн» (Tenney et al., 2019): нижние слои лучше всего кодируют поверхностные и морфологические признаки и части речи, средние — синтаксическую структуру, верхние — семантику, кореференцию и признаки под конкретную задачу. Токен входит в модель как единица текста, а выходит как элемент смысла.

Тот же процесс виден и в декодерах через **logit lens** (nostalgebraist, 2020): промежуточное состояние residual stream можно в любом слое спроецировать выходной unembedding-матрицей и посмотреть, «что модель предсказала бы прямо сейчас». Уже на средних слоях виден грубый черновик будущего токена, который верхние слои постепенно уточняют — предсказание не появляется в конце, а формируется по мере прохождения сигнала.

Отдельные компоненты специализируются. Среди голов внимания находятся позиционные (смотрят на предыдущий токен), синтаксические (следят за парами вроде глагол–дополнение), головы редких токенов; значительную часть остальных голов можно удалить почти без потери качества (Voita et al., 2019; Clark et al., 2019). Знаменитый пример механизма — **induction heads** (Olsson et al., 2022): пара голов в соседних слоях, реализующая копирование по шаблону «...[A][B]...[A] → [B]»; такие головы возникают скачком на раннем этапе обучения и считаются базовым механизмом in-context learning. Важная оговорка: карты внимания показывают, куда переносилась информация, но сами по себе не являются объяснением ответа модели (Jain & Wallace, 2019) — за строгими причинными методами стоит отдельная область механистической интерпретируемости.

## Подсчёт числа параметров

Полезный навык — быстро оценивать размер стандартного decoder-only трансформера. Обозначения: $V$ — размер словаря, $L$ — число блоков, $d$ — размерность модели ($d_{model}$), $d_{ff} = 4d$. Число голов $h$ на счёт не влияет: суммарная размерность голов равна $d$.

Параметры одного блока:

| Компонент | Матрицы | Параметры |
|---|---|---|
| Attention | $W_Q, W_K, W_V, W_O$, каждая $d \times d$ | $4d^2$ |
| FFN | $W_1$: $d \times 4d$, $W_2$: $4d \times d$ | $8d^2$ |
| Нормализации (2 шт.) и bias'ы | векторы длины $d$ | $\sim 10d$, пренебрежимо |

Итого блок $\approx 12d^2$, из них треть — внимание, две трети — FFN. Вне блоков: таблица эмбеддингов $V \cdot d$, обучаемые позиции $n_{ctx} \cdot d$ (если используются) и финальная нормализация; выходная проекция либо связана с эмбеддингами, либо добавляет ещё $V \cdot d$. Итоговая формула:

$$
N \approx 12\,L\,d^2 + V d
$$

Проверим на реальных моделях (словарь GPT-2/GPT-3 — 50257 токенов, эмбеддинги связаны с выходом):

Формула сходится с точностью до долей процента. Заодно видно, как с ростом модели доля эмбеддингов падает с трети (GPT-2 Small) до долей процента (GPT-3): параметры больших моделей почти целиком живут в блоках.

Два замечания о современных вариациях. SwiGLU-FFN содержит три матрицы вместо двух, но с типичным $d_{ff} \approx \tfrac{8}{3}d$ это те же $8d^2$; GQA уменьшает $W_K$ и $W_V$ пропорционально числу K/V-голов. Так что оценка $12Ld^2$ остаётся хорошим приближением. И полезное следствие для следующих глав: forward-проход стоит примерно $2N$ FLOPs на токен, обучение (forward + backward) — примерно $6N$ (Kaplan et al., 2020); связка «параметры → вычисления» — фундамент scaling laws.

## Резюме

Трансформер заменил рекуррентность вниманием и тем самым убрал два главных ограничения RNN: путь между любыми токенами стал константным, а обучение — полностью параллельным по позициям, ценой квадратичной сложности внимания и необходимости явных позиционных кодирований (от синусоид к RoPE и ALiBi). Архитектура собирается из повторяющегося блока «attention + FFN» на residual stream с нормализацией (Pre-LN, RMSNorm). Из неё выросли три семейства: encoder-only (BERT — понимание), decoder-only (GPT — генерация, магистраль современных LLM) и encoder-decoder (T5, BART — задачи вход → выход). Обучение — параллельное предсказание следующего токена под каузальной маской; инференс — авторегрессия с KV-cache, распадающаяся на compute-bound prefill и memory-bound decode, поверх которой работают стратегии сэмплирования от greedy до nucleus. Внутри сигнал движется от поверхностных признаков к синтаксису и семантике, а предсказание постепенно формируется в residual stream. Наконец, размер стандартной модели оценивается формулой $N \approx 12Ld^2 + Vd$ — она пригодится в разговоре о масштабировании.